# Clarkson II event-level feature extraction

This notebook processes the normalized files created by `clarkson2_first3_steps.ipynb`. It pairs key presses with releases and creates one feature row per completed key press/release pair.

Features generated:

- `hold_seconds`: release time minus press time
- `press_to_press_seconds`: current press time minus previous press time
- `release_to_press_seconds`: current press time minus previous release time
- `inter_event_seconds`: current press time minus the previous raw event time

The normalized input and raw dataset are never modified.

In [1]:
from pathlib import Path
from collections import Counter, deque
import csv
import gzip
import json

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)
WORK_DIR = PROJECT_ROOT / 'clarkson2_preprocessing'
INPUT_DIR = WORK_DIR / 'outputs' / 'normalized_events'
OUTPUT_DIR = WORK_DIR / 'outputs' / 'event_features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(INPUT_DIR.glob('*.tsv.gz'))
if not input_files:
    raise FileNotFoundError(f'No normalized files found in {INPUT_DIR}')

print('Input files :', len(input_files))
print('Output dir  :', OUTPUT_DIR)

Input files : 100
Output dir  : C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\event_features


## Pairing policy

A press is stored until a release for the same key arrives. If a key has multiple unmatched presses, releases are matched in first-in-first-out order. Unmatched presses and releases are counted but do not create feature rows.

The timing features are calculated from the event order in the normalized file. Long pauses are retained at this stage; burst segmentation will be handled separately so that the threshold can be selected from the data.

In [2]:
FEATURE_FIELDS = [
    'user_id', 'press_timestamp_100ns', 'release_timestamp_100ns',
    'key', 'hold_seconds', 'press_to_press_seconds',
    'release_to_press_seconds', 'inter_event_seconds'
]

def extract_features(input_path, output_path):
    user_id = input_path.name.removesuffix('.tsv.gz')
    stats = Counter()
    active_presses = {}
    previous_press = None
    previous_release = None
    previous_event = None
    first_timestamp = None

    with gzip.open(input_path, 'rt', encoding='utf-8', newline='') as source, \
         gzip.open(output_path, 'wt', encoding='utf-8', newline='') as destination:
        reader = csv.DictReader(source, delimiter='\t')
        writer = csv.DictWriter(destination, fieldnames=FEATURE_FIELDS,
                                delimiter='\t', lineterminator='\n')
        writer.writeheader()

        for row_number, row in enumerate(reader, start=2):
            stats['input_rows'] += 1
            try:
                timestamp = int(row['timestamp_100ns'])
                flag = row['flag']
                key = row['key']
            except (KeyError, TypeError, ValueError):
                stats['invalid_input_rows'] += 1
                continue

            if first_timestamp is None:
                first_timestamp = timestamp
            if previous_event is not None and timestamp < previous_event:
                stats['timestamp_backsteps'] += 1
            inter_event_seconds = (timestamp - previous_event) / 10_000_000 if previous_event is not None else None

            if flag == '0':
                press_to_press = ((timestamp - previous_press) / 10_000_000
                                  if previous_press is not None else None)
                release_to_press = ((timestamp - previous_release) / 10_000_000
                                    if previous_release is not None else None)
                active_presses.setdefault(key, deque()).append({
                    'timestamp': timestamp,
                    'press_to_press': press_to_press,
                    'release_to_press': release_to_press,
                    'inter_event': inter_event_seconds,
                })
                previous_press = timestamp
                stats['press_events'] += 1
            elif flag == '1':
                queue = active_presses.get(key)
                if not queue:
                    stats['unmatched_release_events'] += 1
                else:
                    press = queue.popleft()
                    hold_seconds = (timestamp - press['timestamp']) / 10_000_000
                    if hold_seconds < 0:
                        stats['negative_hold_pairs'] += 1
                    else:
                        writer.writerow({
                            'user_id': user_id,
                            'press_timestamp_100ns': press['timestamp'],
                            'release_timestamp_100ns': timestamp,
                            'key': key,
                            'hold_seconds': f'{hold_seconds:.7f}',
                            'press_to_press_seconds': '' if press['press_to_press'] is None else f"{press['press_to_press']:.7f}",
                            'release_to_press_seconds': '' if press['release_to_press'] is None else f"{press['release_to_press']:.7f}",
                            'inter_event_seconds': '' if press['inter_event'] is None else f"{press['inter_event']:.7f}",
                        })
                        stats['paired_rows'] += 1
                previous_release = timestamp
                stats['release_events'] += 1
            else:
                stats['invalid_flags'] += 1

            previous_event = timestamp

    stats['unmatched_press_events'] = sum(len(queue) for queue in active_presses.values())
    stats['file'] = user_id
    stats['output'] = str(output_path)
    return dict(stats)

summary = []
for index, input_path in enumerate(input_files, start=1):
    output_path = OUTPUT_DIR / f'{input_path.name.removesuffix(".tsv.gz")}.features.tsv.gz'
    summary.append(extract_features(input_path, output_path))
    if index % 10 == 0 or index == len(input_files):
        print(f'Processed {index}/{len(input_files)} files')

(OUTPUT_DIR / 'feature_extraction_summary.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8')

field_names = sorted({key for row in summary for key in row})
with (OUTPUT_DIR / 'feature_extraction_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=field_names, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(summary)

totals = Counter()
for row in summary:
    for key, value in row.items():
        if key not in {'file', 'output'} and isinstance(value, int):
            totals[key] += value

print('Input events:', totals['input_rows'])
print('Press events:', totals['press_events'])
print('Release events:', totals['release_events'])
print('Paired feature rows:', totals['paired_rows'])
print('Unmatched presses:', totals['unmatched_press_events'])
print('Unmatched releases:', totals['unmatched_release_events'])
print('Negative hold pairs:', totals['negative_hold_pairs'])
print('Feature directory:', OUTPUT_DIR)

Processed 10/100 files


Processed 20/100 files


Processed 30/100 files


Processed 40/100 files


Processed 50/100 files


Processed 60/100 files


Processed 70/100 files


Processed 80/100 files


Processed 90/100 files


Processed 100/100 files
Input events: 25261511
Press events: 14816775
Release events: 10444736
Paired feature rows: 10291192
Unmatched presses: 4391358
Unmatched releases: 19319
Negative hold pairs: 134225
Feature directory: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\event_features


## Output files

- `outputs/event_features/*.features.tsv.gz`: one compressed feature file per user
- `feature_extraction_summary.csv`: per-user pairing and quality counts
- `feature_extraction_summary.json`: machine-readable version of the same report

This notebook does not yet remove long idle periods, clip outliers, create fixed-size windows, normalize features, or create chronological train/adaptation/test splits.